# Tìm lại Parquet M1/M2 trên Drive – Khôi 24127192
Chạy trên Colab bằng tài khoản đã chạy M2 hoặc đã được chia sẻ `m2_full_v5`.
Notebook chỉ đọc nguồn, không crawl và không parse lại. Dừng sau bước tìm kiếm nếu không có bản v5; không thay bằng export M2 cũ.

M1 đã được khôi phục trên máy từ ZIP backup: 5.000 papers và 91.083 references.
Bản v5 theo hướng dẫn nhóm: `IntroDS/milestone2_outputs/m2_full_v5/exports/20261120-203957-9b2330/tables/bibtex`.
Drive chỉ hiện thư mục được tài khoản này sở hữu hoặc có shortcut trong MyDrive. Nếu thiếu thư mục dùng chung, thêm shortcut rồi chạy lại.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os, json, sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas', 'pyarrow'], check=True)
import pyarrow.dataset as ds
import pandas as pd
ROOT = Path('/content/drive/MyDrive')
PAPERS = ['2606.02285','2606.01753','2606.04575','2606.04623','2606.04955']
FOUND = {'v5_runs': [], 'bibtex_v5': [], 'references': [], 'selections': []}
visited = 0
for current, dirs, files in os.walk(ROOT, followlinks=False):
    visited += 1
    p = Path(current)
    if p.name == 'm2_full_v5':
        FOUND['v5_runs'].append(str(p))
        for bib in (p/'exports').glob('*/tables/bibtex'):
            if any(bib.glob('*.parquet')): FOUND['bibtex_v5'].append(str(bib))
        dirs[:] = []
        continue
    if 'references.parquet' in files:
        FOUND['references'].append(str(p/'references.parquet'))
    if p.name == 'references' and any(x.endswith('.parquet') for x in files):
        FOUND['references'].append(str(p))
    if 'selection.csv' in files: FOUND['selections'].append(str(p/'selection.csv'))
    dirs[:] = [d for d in dirs if d not in {'corpus','parsed','.git','node_modules','.venv','elements','hierarchy','bibtex','sources'}]
    if visited >= 30000:
        print('Đã đạt giới hạn 30.000 thư mục; tìm kiếm có thể chưa đầy đủ.')
        break
print(json.dumps(FOUND, ensure_ascii=False, indent=2))
print('Folders scanned:', visited)

## Chọn dữ liệu đã tìm thấy
Điền ba đường dẫn từ output trên. Nếu không thấy `m2_full_v5`, kiểm tra tài khoản Drive/shortcut với người giữ dữ liệu trước. Nếu v5 còn `parsed/` mà mất Parquet, có thể dùng `src/tools/export_verified_m2.py --parsed-dir ... --output-dir <thư mục mới cá nhân>` để xuất lại, không parse lại corpus.

Nếu mất cả v5, cần thống nhất với nhóm việc tạo lại bản chung từ corpus và parser đã chốt; không trộn bản chạy mới vào selection cũ khi chưa kiểm tra provenance.

In [ ]:
BIBTEX = ''       # .../m2_full_v5/exports/<export-id>/tables/bibtex
REFERENCES = ''   # .../2606.00734-2606.05733/work/tables/references.parquet hoặc m1_tables
SELECTION = ''    # .../m3_selection_v3/selection.csv
if not all([BIBTEX, REFERENCES, SELECTION]):
    raise ValueError('Điền BIBTEX, REFERENCES và SELECTION từ kết quả tìm kiếm')
if 'm2_full_v5' not in Path(BIBTEX).parts:
    raise ValueError('Chưa chọn bibtex từ bản m2_full_v5 đã chốt')
selection = pd.read_csv(SELECTION, dtype=str, keep_default_na=False)
mine = selection[(selection.owner == '24127192') & (selection.kind == 'manual')]
if len(mine) != 5 or set(mine.arxiv_id) != set(PAPERS):
    raise ValueError('Selection không khớp 5 bài manual đã chốt')
small = {}
for name, path in [('bibtex',BIBTEX),('references',REFERENCES)]:
    dataset = ds.dataset(path, format='parquet')
    table = dataset.to_table(filter=ds.field('arxiv_id').isin(PAPERS))
    counts = table.to_pandas().groupby('arxiv_id').size().to_dict()
    if set(counts) != set(PAPERS):
        raise ValueError(f'{name}: thiếu dữ liệu cho một trong 5 bài: {counts}')
    small[name] = table
    print(name, counts)

## Tải gói nhỏ về máy để tiếp tục manual
Gói chứa BibTeX và references thật của 5 bài, cùng selection đã chốt. Chưa có nhãn manual. Gửi đường dẫn file ZIP tải về trong chat để tiếp tục tạo sheets và rà soát. Không cần tải toàn bộ corpus hoặc hàng triệu dòng elements.

In [ ]:
import tempfile, shutil, hashlib
import pyarrow.parquet as pq
from google.colab import files
out = Path(tempfile.mkdtemp(prefix='m3_khoi_inputs_', dir='/content'))
for name, table in small.items():
    pq.write_table(table, out/f'{name}.parquet', compression='zstd')
shutil.copy2(SELECTION, out/'selection.csv')
manifest = {'owner':'24127192','papers':PAPERS,'bibtex_source':BIBTEX,'references_source':REFERENCES,
            'selection_source':SELECTION,'selection_sha256':hashlib.sha256(Path(SELECTION).read_bytes()).hexdigest(),
            'status':'source_subset_only_no_manual_labels'}
(out/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2))
archive = shutil.make_archive(str(out), 'zip', out)
print(archive)
files.download(archive)